# BERT Approach

This notebook presents the bert based approach in solving the three way classification problem of clarity labeling

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from torch.optim import AdamW
import tqdm


### This only applies if user is using Google Drive ###

from google.colab import drive
import os

drive.mount('/content/drive')


# This assumes you have a folder named 'Colab_Project' in your 'My Drive'
DRIVE_PATH = '/content/drive/MyDrive/dataset'

os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Drive mounted and directory checked at: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and directory checked at: /content/drive/MyDrive/dataset


In [2]:

dataset_path = f'{DRIVE_PATH}/test_data.csv'
df = pd.read_csv(dataset_path)

# drop_cols = ['evasion_label', 'Unnamed: 0', 'affirmative_questions']
# df = df.drop(columns= drop_cols)
df.head()

,title,date,president,url,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,annotator_id,annotator1,annotator2,annotator3,inaudible,multiple_questions,affirmative_questions,index,clarity_label,evasion_label
0,NaN,NaN,NaN,https://www.presidency.ucsb.edu/documents/the-...,5,"Q. What about the redline, sir?","Well, the world has made it clear that these t...",NaN,NaN,Inquiring about the status or information reg...,NaN,Dodging,General,Dodging,False,False,True,0,Ambivalent,NaN
1,NaN,NaN,NaN,https://www.presidency.ucsb.edu/documents/the-...,2,Q. Will you invite them to the White House to ...,I think that anytime and anyplace that they ar...,NaN,NaN,Will you invite them to the White House to neg...,NaN,Deflection,General,General,False,False,False,1,Ambivalent,NaN
2,NaN,NaN,NaN,https://www.presidency.ucsb.edu/documents/the-...,1,"Q. Harsh. Mr. President, Japan has dropped the...",I think that the purpose of the U.N. Security ...,NaN,NaN,Why was it necessary for Japan to drop the thr...,NaN,Explicit,Implicit,Implicit,False,False,False,2,Ambivalent,NaN
3,NaN,NaN,NaN,https://www.presidency.ucsb.edu/documents/the-...,2,Q. The Lebanese Prime Minister is demanding a ...,I'll let Condi talk about the details of what ...,NaN,NaN,When will we see this resolution?,NaN,Explicit,General,General,False,False,False,3,Ambivalent,NaN
4,NaN,NaN,NaN,https://www.presidency.ucsb.edu/documents/the-...,7,"Q. Thank you, Mr. President. Back on Iraq, a g...","No, I don't consider it a credible report; nei...",NaN,NaN,Updating the figure of Iraqi deaths,NaN,Dodging,Implicit,Dodging,False,False,True,4,Ambivalent,NaN


In [2]:
label_map = {
    'Clear Reply': 0,
    'Clear Non-Reply': 1,
    'Ambivalent': 2
}
num_labels = len(label_map)
model_name = 'bert-base-uncased'


modelBert = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) # For binary classification
tokenizerBert = BertTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
def data_prep(questions, answers, clarity_label):
    encodings = tokenizerBert(questions.tolist(),
                          answers.tolist(),
                          padding= True,
                          truncation = True,
                          max_length = 512,
                          return_tensors = 'pt'
                          )
    label_tensors = torch.tensor([label_map[l] for l in clarity_label.tolist()])
    dataset = TensorDataset(
        encodings['input_ids'],
        encodings['attention_mask'],
        encodings['token_type_ids'],
        label_tensors
        )
    return dataset


kör :

pip install -U bitsandbytes transformers accelerate


In [5]:
from huggingface_hub import login

login()

In [6]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=True
)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [7]:
SYSTEM_PROMPT = """
You are a highly accurate text extraction assistant. Your task is to analyze a long interview answer and a specific subquestion.

You must identify the exact sentence or sentences within the provided interview answer that directly address the subquestion.

If the subquestion is not answered, return only the phrase "NOT_ANSWERED".

Do not paraphrase, summarize, or add any extra commentary. Return only the extracted text or the "NOT_ANSWERED" phrase.
"""

SYSTEM_PROMPT_SUMMARY = """
You are a precise extraction and summarization assistant.

You receive:
- A subquestion (Question)
- A long interview answer (Answer)

Task:
1. Find only the part(s) of the Answer that directly address the Question.
2. Ignore unrelated digressions.
3. Write a concise, neutral summary of that relevant part only.

Output format (exactly):
<your summary>
"""

def get_answers_llama3(question, full_answer, SYSTEM_PROMPT = SYSTEM_PROMPT):
    user_prompt = f"""
Subquestion: "{question}"

Full Interview Answer:
---
{full_answer}
---

Summary:
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_tensors = model.generate(
            model_inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    output_text = tokenizer.decode(output_tensors[0][model_inputs.shape[1]:], skip_special_tokens=True).strip()
    return output_text

# Example test before running on the full dataframe
test_q = "What is the policy?"
test_a = "Our policy is clearly defined in section 3. We updated it last year after feedback."
print(f"Extracted: {get_answers_llama3(test_q, test_a, SYSTEM_PROMPT_SUMMARY)}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Extracted: "Our policy is clearly defined in section 3."


In [8]:
import os
from tqdm.auto import tqdm

processed_df = df.copy()
processed_df['answer'] = None # Initialize a new column

# Define where to save checkpoints
CHECKPOINT_FILE = f'{DRIVE_PATH}/ckpt_test_summary.csv'
CHECKPOINT_INTERVAL = 200
DISPLAY_INTERVAL = 100

if os.path.exists(CHECKPOINT_FILE):
    print(f"Checkpoint found. Resuming from {CHECKPOINT_FILE}")
    processed_df = pd.read_csv(CHECKPOINT_FILE)
    last_processed_index = processed_df['answer'].last_valid_index()
    if last_processed_index is not None:
        start_index = last_processed_index + 1
    else:
        start_index = 0
else:
    start_index = 0


print(f"Starting extraction from entry {start_index} of {len(processed_df)}")

for index, row in tqdm(processed_df.iloc[start_index:].iterrows(), total=len(processed_df.iloc[start_index:])):

    question = row['question']
    interview_answer = row['interview_answer']

    extracted_text = get_answers_llama3(question, interview_answer, SYSTEM_PROMPT_SUMMARY)

    processed_df.at[index, 'answer'] = extracted_text

    if (index + 1) % DISPLAY_INTERVAL == 0:
        print(f"\n--- Sample Display at Entry {index + 1} ---")
        print(f"Question: {question[:100]}...")
        print(f"Full Answer (Snippet): {interview_answer[:150]}...")
        print(f"**Extracted Answer:** {extracted_text}")
        print("------------------------------------------")


    if (index + 1) % CHECKPOINT_INTERVAL == 0:
        processed_df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"\nCheckpoint saved at entry {index + 1}.")

print("Processing complete!")

FINAL_FILE_NAME = 'training_data_preprocessed_LLM_summary.csv'
processed_df.to_csv(FINAL_FILE_NAME, index=False)
print(f"Final data saved to {FINAL_FILE_NAME}")


Starting extraction from entry 0 of 308


  0%|          | 0/308 [00:00<?, ?it/s]


--- Sample Display at Entry 100 ---
Question: What kind of punishment would you like to see imposed on North Korea, short of some sort of condemna...
Full Answer (Snippet): Okay....
**Extracted Answer:** "Economic sanctions, such as cutting off trade and restricting access to international financial systems, would be an effective punishment for North Korea, as it would severely impact their ability to fund their military and nuclear programs."
------------------------------------------

--- Sample Display at Entry 200 ---
Question: Do you agree with the Prime Minister's supporters that he led the way on the issue?...
Full Answer (Snippet): Well, on your second question, Mr. McKinnon, we have proceeded through all the processes required under our extradition agreements. It is now in the h...
**Extracted Answer:** The relevant part of the answer that directly addresses the question "Do you agree with the Prime Minister's supporters that he led the way on the issue?" is:

"I totally unde

In [4]:

from sklearn.model_selection import train_test_split

EVASION = True
CLARITY_LABEL = 'Ambivalent'


### Paths need to be modified to fit the setup of the user ###
### Paths can be adjusted according to dataset preference ###
path_summary = '/content/training_data_preprocessed_LLM_summary.csv'
path_exctracted = '/content/drive/MyDrive/dataset/training_data_preprocessed_LLM.csv'
path_regular = '/content/drive/MyDrive/dataset/training_data_processed.csv'

df = pd.read_csv(path_regular)

if EVASION:
  df = df[df["clarity_label"] == CLARITY_LABEL]
  if CLARITY_LABEL == 'Ambivalent':
    label_map = {
        'Implicit': 0,
        'Dodging': 1,
        'General': 2,
        'Deflection':3,
        'Partial/half-answer':4
    }
  elif CLARITY_LABEL == 'Clear Non-Reply':
    label_map = {
        'Declining to answer': 0,
        'Claims Ignorance': 1,
        'Clarification': 2,
    }

Q_train, Q_val, A_train, A_val, L_train, L_val = train_test_split(
    df['question'],
    df['interview_answer'],
    df['evasion_label'], ## change to 'clarity-label' for task one
    test_size=0.2,
    random_state=40
    )
BATCH_SIZE = 16

train_dataset = data_prep(Q_train, A_train, L_train)
val_dataset = data_prep(Q_val, A_val, L_val)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

### Model Training

With preprocessing done we can now gop on to build the training loop to finetune the BERT model

In [5]:
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report



model = modelBert

def eval_model(model, data_loader, device):
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, token_type_ids, labels = [t.to(device) for t in batch]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            _, predicted = torch.max(outputs.logits, 1)


            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    model.train()

    accuracy = 100 * np.sum(np.array(all_predictions) == np.array(all_labels)) / len(all_labels)
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    conf_matrix = confusion_matrix(all_labels, all_predictions)

    class_report = classification_report(all_labels, all_predictions, digits=4, output_dict=True)

    return accuracy, f1_weighted, conf_matrix, class_report


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
epochs = 5
optimizer = AdamW(model.parameters(), lr = 5e-5)



for epoch in range(epochs):
    print(f"--- Starting Epoch {epoch+1}/{epochs} ---")
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        token_type_ids = batch[2].to(device)
        labels = batch[3].to(device)

        model.zero_grad()

        outputs = model(input_ids= input_ids,
                        attention_mask = attention_mask,
                        token_type_ids = token_type_ids,
                        labels = labels
                        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    val_accuracy, val_f1, val_conf_matrix, val_report = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1} Summary:")
    print(f"  Training Loss: {avg_train_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.2f}%")
    print(f"  Validation F1 (Weighted): {val_f1:.4f}")

print("\n  --- Confusion Matrix (Validation) ---")
labels_list = list(label_map.keys())
cm_df = pd.DataFrame(val_conf_matrix, index=labels_list, columns=labels_list)
print(cm_df)

--- Starting Epoch 1/5 ---
